In [1]:
# %%
import polars as pl
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"
pipeline_dir = data_dir / "Pipeline"

RUN_FOLDERS = {
    "ranking": "10_all_CP_complexes",
    "pae": "18_CP_model_based_on_pae",
    "trimmed": "19_CP_pae_plddt_trimmed",
    "trimmed_safeguard": "20_CP_pae_plddt_trimmed_interface_safeguard",
}
mapping_path = data_dir / "complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv"
complextab_path = data_dir / "Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv"
N_PROTEINS_ABOVE = 2

In [2]:
# %%
def canonical_stoich(expr: pl.Expr) -> pl.Expr:
    """'B:1,A:2' -> 'A:2,B:1' so stoichiometries can be compared as strings"""
    return expr.str.split(",").list.sort().list.join(",")


def predicted_stoich(column: str) -> pl.Expr:
    """pred cell '{"P1":1,"P2":2},{"rank":1,...}' -> 'P1:1,P2:2' (sorted)"""
    return canonical_stoich(pl.col(column).str.extract(r"^\{([^}]*)\}").str.replace_all('"', ""))


def best_confidence(column: str) -> pl.Expr:
    """'{0: 65.5, 1: 68.0}' -> 68.0; '{}' -> null"""
    return (pl.col(column).str.extract_all(r": [\d.]+")
            .list.eval(pl.element().str.slice(2).cast(pl.Float64)).list.max())


def read_combfold_results(folder: str) -> pl.DataFrame:
    path = pipeline_dir / folder / f"all_pdb_present_{folder}_pool_pipeline_complexes_combfold_results.csv"
    assert path.exists(), f"missing results file: {path}"
    combfold_results = pl.read_csv(path)
    assert combfold_results["complex_ac"].is_unique().all(), f"{folder}: more than one row per complex"
    return combfold_results


combfold_results_per_run = {run: read_combfold_results(folder) for run, folder in RUN_FOLDERS.items()}
for run, run_results in combfold_results_per_run.items():
    print(f"{run}: {run_results.height} complexes")

ranking: 613 complexes
pae: 466 complexes
trimmed: 466 complexes
trimmed_safeguard: 466 complexes


In [3]:
# %%
# complexes with at least one predicted stoichiometry that produced an assembly, per run
complexes_with_output_per_run = {
    run: set(
        run_results
        .with_columns(best_pred_confidence=pl.max_horizontal(
            best_confidence("CF_1_confidence"), best_confidence("CF_2_confidence"), best_confidence("CF_3_confidence")))
        .filter(pl.col("best_pred_confidence").is_not_null())["complex_ac"]
    )
    for run, run_results in combfold_results_per_run.items()
}
for run, complexes in complexes_with_output_per_run.items():
    print(f"{run}: {len(complexes)} complexes with an output")

# match class + protein count
mapping = pl.read_csv(mapping_path)
complex_properties = mapping.select("complex_ac", "match_class", "n_proteins").unique()
assert complex_properties["complex_ac"].is_unique().all(), "a complex has several match_class / n_proteins rows"

# true stoichiometry from the Complex Portal tsv
complextab_raw = pl.read_csv(complextab_path, separator="\t")
true_stoich_per_complex = (
    complextab_raw
    .rename({"#Complex ac": "complex_ac", complextab_raw.columns[4]: "identifiers"})
    .select(
        "complex_ac",
        true_stoich=canonical_stoich(pl.col("identifiers").str.replace_all(r"\((\d+)\)", ":$1").str.replace_all(r"\|", ",")),
        stoich_is_known=~pl.col("identifiers").str.contains(r"\(0\)"),
    )
)
assert true_stoich_per_complex["complex_ac"].is_unique().all(), "duplicate complex_ac in the ComplexTab"

ranking: 553 complexes with an output
pae: 414 complexes with an output
trimmed: 407 complexes with an output
trimmed_safeguard: 415 complexes with an output


In [4]:
# %%
# complexes whose pred_1/2/3 stoichiometries are identical in all 4 runs
pred_stoich_per_run = pl.concat([
    run_results.select("complex_ac", run=pl.lit(run), pred=pl.lit(f"pred_{i}"), stoich=predicted_stoich(f"pred_{i}"))
    for run, run_results in combfold_results_per_run.items()
    for i in (1, 2, 3)
])

n_distinct_stoich_per_pred = pred_stoich_per_run.group_by("complex_ac", "pred").agg(n_distinct=pl.col("stoich").n_unique())
complexes_with_identical_preds = set(
    n_distinct_stoich_per_pred
    .group_by("complex_ac").agg(all_preds_identical=(pl.col("n_distinct") == 1).all())
    .filter(pl.col("all_preds_identical"))["complex_ac"]
)
print(f"{len(complexes_with_identical_preds)} complexes with identical pred_1/2/3 in all runs")

# %%
ranking_complexes = combfold_results_per_run["ranking"]
funnel_steps = [("ranking csv: all complexes", ranking_complexes.height)]

# 1) output in ranking, pae, trimmed and trimmed_safeguard
complexes_with_output_in_all_runs = set.intersection(*complexes_with_output_per_run.values())
ranking_complexes = ranking_complexes.filter(pl.col("complex_ac").is_in(list(complexes_with_output_in_all_runs)))
funnel_steps.append(("output in all 4 runs", ranking_complexes.height))

# 1b) pred_1/2/3 identical in all 4 runs (same filter as the heatmap)
ranking_complexes = ranking_complexes.filter(pl.col("complex_ac").is_in(list(complexes_with_identical_preds)))
funnel_steps.append(("pred_1/2/3 identical across runs", ranking_complexes.height))

# 2) benchmark = exact_pdb_match
ranking_complexes = ranking_complexes.join(complex_properties, on="complex_ac", how="left")
print(f"{ranking_complexes['match_class'].null_count()} complexes not found in the mapping (they drop out here)")
ranking_complexes = ranking_complexes.filter(pl.col("match_class") == "exact_pdb_match")
funnel_steps.append(("benchmark (exact_pdb_match)", ranking_complexes.height))

# 3) more than 2 proteins
ranking_complexes = ranking_complexes.filter(pl.col("n_proteins") > N_PROTEINS_ABOVE)
funnel_steps.append((f"n_proteins > {N_PROTEINS_ABOVE}", ranking_complexes.height))

# 4) stoichiometry given (no "(0)")
ranking_complexes = ranking_complexes.join(true_stoich_per_complex, on="complex_ac", how="left")
print(f"{ranking_complexes['true_stoich'].null_count()} complexes not found in the ComplexTab (they drop out here)")
ranking_complexes = ranking_complexes.filter(pl.col("stoich_is_known").fill_null(False))
funnel_steps.append(("stoichiometry known (no '(0)')", ranking_complexes.height))

print(pl.DataFrame(funnel_steps, schema=["step", "n_complexes"], orient="row"))

603 complexes with identical pred_1/2/3 in all runs
0 complexes not found in the mapping (they drop out here)
0 complexes not found in the ComplexTab (they drop out here)
shape: (6, 2)
┌─────────────────────────────────┬─────────────┐
│ step                            ┆ n_complexes │
│ ---                             ┆ ---         │
│ str                             ┆ i64         │
╞═════════════════════════════════╪═════════════╡
│ ranking csv: all complexes      ┆ 613         │
│ output in all 4 runs            ┆ 394         │
│ pred_1/2/3 identical across ru… ┆ 387         │
│ benchmark (exact_pdb_match)     ┆ 123         │
│ n_proteins > 2                  ┆ 62          │
│ stoichiometry known (no '(0)')  ┆ 44          │
└─────────────────────────────────┴─────────────┘


In [5]:
# %%
ranking_with_matches = ranking_complexes.with_columns(
    stoich_pred_1=predicted_stoich("pred_1"),
    stoich_pred_2=predicted_stoich("pred_2"),
    stoich_pred_3=predicted_stoich("pred_3"),
).with_columns(
    true_is_pred_1=pl.col("stoich_pred_1") == pl.col("true_stoich"),
    true_is_pred_2=pl.col("stoich_pred_2") == pl.col("true_stoich"),
    true_is_pred_3=pl.col("stoich_pred_3") == pl.col("true_stoich"),
).with_columns(
    true_among_preds=pl.any_horizontal("true_is_pred_1", "true_is_pred_2", "true_is_pred_3"),
    # the pipeline's own verdict: rank of the prediction that matched, or "none"
    pipeline_says_true_among_preds=pl.col("correct_pred_rank").cast(pl.String) != "none",
)

print(f"my string match:        true among preds in {ranking_with_matches['true_among_preds'].sum()} / {ranking_with_matches.height}")
print(f"pipeline's rank column: true among preds in {ranking_with_matches['pipeline_says_true_among_preds'].sum()} / {ranking_with_matches.height}")
print(ranking_with_matches["correct_pred_rank"].cast(pl.String).value_counts().sort("correct_pred_rank"))

n_disagree = ranking_with_matches.filter(pl.col("true_among_preds") != pl.col("pipeline_says_true_among_preds")).height
print(f"complexes where my match and the pipeline disagree: {n_disagree}")

# %%
# a look at complexes where the true stoichiometry is not among the predictions
with pl.Config(tbl_rows=30, fmt_str_lengths=120, tbl_width_chars=250):
    print(
        ranking_with_matches
        .filter(~pl.col("true_among_preds"))
        .select("complex_ac", "true_stoich", "stoich_pred_1", "stoich_pred_2", "stoich_pred_3")
        .head(15)
    )

my string match:        true among preds in 21 / 44
pipeline's rank column: true among preds in 33 / 44
shape: (4, 2)
┌───────────────────┬───────┐
│ correct_pred_rank ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ 1                 ┆ 29    │
│ 2                 ┆ 2     │
│ 3                 ┆ 2     │
│ none              ┆ 11    │
└───────────────────┴───────┘
complexes where my match and the pipeline disagree: 12
shape: (15, 5)
┌────────────┬──────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────┐
│ complex_ac ┆ true_stoich                                              ┆ stoich_pred_1                                            ┆ stoich_pred_2                                            ┆ stoich_pred_3                                            │
│ ---     